# Exercise: Segmentation and semantic segmentation

In this exercise we first will look at the SLIC and GraphCuts approaches to segmentation and then continue with some methods exploring semantic segmentation.

In [ ]:
import numpy as np
import skimage
import skimage.data
import skimage.io
import skimage.future
import skimage.segmentation
import sklearn.cluster
import cv2

# for displaying images in jupyter
from matplotlib import pyplot as plt

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 10]

## Exercise 1: Superpixels and GraphCuts

We have discussed the method of superpixel segmentation using the SLIC algorithm as a first step in graphcuts segmentation. There is an implementation at

https://scikit-image.org/docs/dev/api/skimage.segmentation.html#skimage.segmentation.slic

We will use an image from a dataset for autonomous vehicles for this exercise.

In [ ]:
image = skimage.io.imread('/exchange/cvai/images/000041_10.png')
plt.imshow(image)

Use the SLIC implementation to calculate the superpixels and save the result in a variable called `labels`. This will contain the labels of each region. Look at the parameters of the method and play with the number of regions. The code in the next cell will display the result using labels2rgb.

In [ ]:
labels = None
# YOUR CODE HERE
raise NotImplementedError()


In [ ]:
plt.imshow(skimage.color.label2rgb(labels, image, kind='avg', bg_label=-1))

Next we would like to further segment the image using the graph cut algorithm. In order to do this we must first convert the image, respectively the labels from the SLIC segmentation into a Region Adjacency Graph (RAG).

This will calculate weights between the regions using the mean color of the regions:
http://scikit-image.org/docs/dev/api/skimage.graph.html#skimage.graph.rag_mean_color



In [ ]:
graph = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# We can actually display the graph overlayed on the image
skimage.graph.show_rag(labels, graph, image, border_color=None)

Once the graph is constructed we can use the graph cut algorithm:

https://scikit-image.org/docs/stable/api/skimage.graph.html#skimage.graph.cut_normalized

It will return the new set of labels.

In [ ]:
labels_cut = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
assert labels_cut is not None
plt.imshow(skimage.color.label2rgb(labels_cut, image, kind='avg', bg_label=-1))

## Exercise 2: Semantic Segmentation

Semantic segmentation is the process of classifying each image pixel into one of several classes. In the exercise we will look at some examples how the features can be calculated for this. 

We will use two sample images from a dataset for texture classes, the *describable texture database (dtd)*:

https://www.robots.ox.ac.uk/~vgg/data/dtd/

In [ ]:
im_1 = skimage.io.imread('/exchange/cvai/images/crosshatched_0044.jpg')
im_2 = skimage.io.imread('/exchange/cvai/images/waffled_0029.jpg')
plt.rcParams['figure.figsize'] = [10, 10]
plt.subplot(2, 1, 1)
plt.imshow(im_1)
plt.subplot(2, 1, 2)
plt.imshow(im_2)

### Use grayscale images

We will only use the gray images, so we will convert them to grayscale. By default, this will turn them into float64 images if we use the ```skimage.color.rgb2gray``` function. As we want to continue to work with 8bit images, we convert them back to ubyte using ```skimage.img_as_ubyte```

In [ ]:
im_1_gray = skimage.img_as_ubyte(skimage.color.rgb2gray(im_1))
plt.imshow(im_1_gray, cmap='gray')

In [ ]:
# do the same for the other image
im_2_gray = None
# YOUR CODE HERE
raise NotImplementedError()

### Sliding window approach

We have seen in the lecture, that we should calculate features from a neighborhood around the image and then go to the next pixel and so on. This is a sliding window approach. 

Insteading of stepping through a double for loop, skimage provides two utility function to calculate blocks from an image. One is `skimage.util.view_as_blocks` that divides the image into non-overlapping blocks and the other is `skimage.util.view_as_windows` that divides the image into overlapping windows. For the latter approach, we have to be careful to not generate too many windows, so in the sliding window approach we often use some step value greater than 1.

Use the function `skimage.util.view_as_blocks` and see how to access the blocks. Plot the blocks as images. Use a blocksize of 80 by 80 pixels.



In [ ]:
blocks_1 = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
np.testing.assert_array_equal(blocks_1.shape, (8,8,80,80))

We can now calculate some features for each block that should be characteristic for the region. We will use the GLCM (using skimage.feature.graycomatrix). The matrix is a square nxn matrix, where n is the number of levels. For uint8 images n would be 256, however this is probably too large, so we use only 16 levels which can be achieved by dividing the image values by 16 (using the // operator in python for integer division).


Calculate a matrix and indicate the correct number of levels. Check the documentation about the arguments.

In [ ]:
# the blocks above calculated above is a 2D array of 80x80 blocks, however wo dont really need the 2D structure 
# anymore and would like to have just an 1D array of 80x80 blocks
blocks_1_flat = blocks_1.reshape(-1, blocks_1.shape[2], blocks_1.shape[3])

# the first block can then be accessed by blocks_1_flat[0], calculate the GLCM of this
glcm = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
np.testing.assert_array_equal(glcm.shape, (16,16,1,1))

#### Multiple GLCMs

It is possible to calculate different GLCMs for different pixel neighborhoods and different orientations. Change the arguments to calculate GLCMs for the distances 1 and 2 pixels and for horizontal and vertical orientations.

In [ ]:
glcm = None
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
np.testing.assert_array_equal(glcm.shape, (16,16,2,2))

It is better to use properties calculated from GLCM instead of using the properties directly. Features can be calculated using `skimage.feature.graycoprops` such as contrast and dissimilarity. The full list of supported features is

`['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']`

Calculate those features and print them.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

### Final feature calculation

Define a function that takes as input an image (or a block) and then calculate the features by first calculating the GLCM (using 2 distances and directions), then calculating the properties and finally putting everything into one feature vector and returning it.

In [ ]:
def calc_features(image: np.ndarray) -> np.ndarray:
    features = []
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
f = calc_features(blocks_1_flat[0] // 16)
print(f)
assert f.shape[0] == 24

We can now put together a data set from the features of the 2 images, we use a smaller block size to get more data. Of course, normally we would have many different images for each class and divide the data set into train, test and validation, but here we want to concentrate on the feature extraction to see how this would work.

In [ ]:
blocks_1 = skimage.util.shape.view_as_blocks(im_1_gray, block_shape=(20,20))
blocks_1_flat = blocks_1.reshape(-1, blocks_1.shape[2], blocks_1.shape[3])
f_1 = [calc_features(image // 16) for image in blocks_1_flat]
data_1 = np.stack(f_1, axis=0)
label_1 = np.empty(data_1.shape[0])
label_1.fill(0)

blocks_2 = skimage.util.shape.view_as_blocks(im_2_gray[0:420,:], block_shape=(20,20))
blocks_2_flat = blocks_2.reshape(-1, blocks_2.shape[2], blocks_2.shape[3])
f_2 = [calc_features(image // 16) for image in blocks_2_flat]
data_2 = np.stack(f_2, axis=0)
label_2 = np.empty(data_2.shape[0])
label_2.fill(1)

# create a full data set
data = np.concatenate([data_1, data_2])
label = np.concatenate([label_1, label_2])

In [ ]:
print(data.shape)

We can now train a classifier on those features. We will use a decision tree for this for a change, but an SVM would work too, of course.

In [ ]:
# use a decision tree classifier
dtc = sklearn.tree.DecisionTreeClassifier()
dtc.fit(data, label)
# prediction
labels_p = dtc.predict(data)
sklearn.metrics.confusion_matrix(label, label)

The regions have quite different characteristics, so the training accuracy is 1.0. 